# W8T41JZK0ZMEP

Packages

In [ ]:
# Public packages
import math
import os
import gc
import re
from tqdm import tqdm
import tabulate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown
from pathlib import Path

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.labeling_functions import fully_relabel_and_consolidate, plot_dish_time_series, rename_items
from tools.coverage_functions import plot_time_series

# Preemptively set new Pandas option, increase display max rows, allow matplotlib visuals, and set matplotlib to close
pd.options.mode.copy_on_write = True
pd.set_option('display.max_rows', 100)
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
%load_ext autoreload
%autoreload 2

# Load formatted data if it's there 
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files (ignore details)

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id').assign(cross_over_date = lambda df: pd.to_datetime(df['cross_over_date']))


# Timezones
timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])
    sales_and_menu_data[loc_id] = df


restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()


loc_id = 'W8T41JZK0ZMEP'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
df_uncleaned.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan")')[['item_name','item_modifications']].value_counts()

In [ ]:
print(df_uncleaned.query('is_plant_based == "No"')['item_name'].value_counts().to_string())

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Vegan Breakfast Sandwich")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Vegan Breakfast Sandwich")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

In [ ]:
df_uncleaned.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cupcake")').index[0]

In [ ]:
df_uncleaned.loc[:pd.Timestamp('2020-04-27 13:28:58-0400', tz='America/New_York')]

# Type the entries with modifications here:

In [ ]:
# Original item name, animal-based ingredient, new name for item , custom/
# 0 for standard, 1 for custom
modification_name_changes = [ # Standard and Custom
    (('Avocado Toast','Egg','Egg Avocado Toast'),('0')), 
    (('Pb & J','Collagen','Collagen Pb & J'), ('1')),
    (('Collagen Pb & J','Collagen, Vegan','Pb & J'), ('Custom')),
    (('Morning Mocha','Collagen|Whipped Cream','Whipped Cream Collagen Morning Mocha'), ('Custom')),
    (('Whipped Cream Collagen Morning Mocha','Collagen, Vegan|Whipped Cream, Non-Dairy','Morning Mocha'), ('Custom')),
    (('Acai','Collagen','Collagen Acai'), ('Custom')),
    (('Pb & J Bowl','Collagen','Collagen Pb & J Bowl'), ('Custom')),
    (('Collagen Pb & J Bowl','Collagen, Vegan','Pb & J Bowl'), ('Custom')),
    (('Vegan Breakfast Sandwich','Egg|Regular Egg|Regular Cheese|Reg Egg|Real Scrambled Egg|Real Egg|Egg White','Veg Breakfast Sandwich'), ('Custom')),
    (('Acai Smoothie Bowl','Collagen, Regular','Collagen Acai Smoothie Bowl'), ('Custom')),
    (('Blueberry Thrill','Collagen, Regular','Collagen Blueberry Thrill'), ('')),
    (('Avocado Toast','Egg|Sunny','Egg Avocado Toast'), ('')),
    (('Buddha Bowl','Chicken','Chicken Buddha Bowl'), ('')),
    (('Buddha Bowl','Egg',''), ('Egg Buddha Bowl')),
    (('Beyond Burger','Regular Cheese|Reg Cheese','Cheese Beyond Burger'), ('')),
    (('Avocado Toast - Gf Available','Egg','Egg Avocado Toast'), ('')), 
    (('Salads','Chicken','Chicken Salads'), ('')),
    (('Great Pumpkin Bowl','Dairy-Regular','Great Pumpkin Bowl (Dairy)'), ('')),
    (('Kale!  Ceasar Salad','Regular','Regular Kale! Cesar Salad'), ('')),
    (('Veggie Melt & Tomato Soup','Vegan','Vegan Veggie Melt & Tomato Soup'), ('')),
    (('Veggie Melt & Tomato Basil Soup','Vegan','Vegan Veggie Melt & Tomato Basil Soup'), ('')),
    (('Goddess Salad','Chicke','Chicken Goddess Salad'), ('')),
    (('Goddess Salad','Egg','Egg Goddess Salad'), ('')),
    (('Sw Burrito Wrap','Chicken|Egg',''), ('Chicken Sw Burrito Wrap')),
    (('Reuben-Vegan Or Regular','Turkey',''), ('Turkey Reuben-Vegan Or Regular')),
    (('Spinach Salad','Organic Chicken|Feta Cheese','Chicken Spinach Salad'), ('')),
    (('Apple Walnut Salad','Chicken|Egg','Chicken Apple Walnut Salad'), ('')),
    (('Green Goddess Salad Bowl','Chicken|Egg','Chicken Green Goddess Salad Bowl'), ('')),
    (('Reuben-Vegan(Tofu) Or Rachel(Turkey)','Turkey|Cheese','Turkey Reuben-Vegan(Tofu) Or Rachel(Turkey)'), ('')),
    (('Breakfast Bowl','Egg|Regular Cheddar|Regular Cheese|Mayo','Cheesy Breakfast Bowl W Egg '), ('')),
    (('Breakfast Bowl','Vegan','Vegan Breakfast Bowl'), ('')),
    (('Breakfast Wrap','Egg|Regular Cheese|Mayo','Eggy & Cheezy Breakfast Wrap'), ('')),
    (('Eggy & Cheezy Breakfast Wrap','No Egg','Breakfast Wrap'), ('')),
    (('Breakfast Wrap','Vegan','Vegan Breakfast Wrap'), ('')),
    (('Toasted Quesadilla','Organic Chicken|Regular Sour Cream|Regular  Vegan Cheddar Cheese','Chicken/Sour Cream Toasted Quesadilla'), ('')),
    (('Toasted Quesadilla','Vegan Cheddar Cheese|Vegan Sour Cream','Vegan Toasted Quesadilla'), ('')),
    (('Sofritas Quesadilla','Vegan Cheddar','Vegan Cheddar Sofritas Quesadilla'), ('')),
    (('Quesadilla','Vegan',''), ('Vegan Quesadilla')),
    (('Grilled Cheese & Tomato Soup','Vegan','Vegan Grilled Cheese & Tomato Soup'), ('')),
    (('Hot Soup(V/Gf) With Grilled Turkey And Cheese (Vegan, Gf Option By Request)','No Turkey','Vegan Hot Soup'), ('')),
    (('Cheese Melt & Soup','Vegan','Vegan'), ('Vegan Cheese Melt & Soup')),
    (('Soup & Grilled Cheese','Vegan Cheddar|Vegan Prov','Soup & Grilled Vegan Cheese'), ('')),
    (('Egg Bakes','Chicken','Chicken Egg Bakes'), ('')),
    (('Autumn Salad Bowl','Chicken','Chicken Autumn Salad Bowl'), ('')),
    (('Cauliflower Flatbread Pizza','Chicken','Chicken Cauliflower Flatbread Pizza'), ('')),
    (('Cauliflower Flatbread Pizza','Vegan','Vegan Cheese Cauliflower Flatbread Pizza'), ('')),
    (('Chicken Salad Over Greens','Chickpea','Chickpea Salad Over Greens'), ('')),
    (("Farmer'S Market Salad","Chicken","Chicken Farmer'S Market Salad"), ('')),
    (("Farmer'S Market Salad","Vegan","Vegan Farmer'S Market Salad"), ('')),
    (("Wraps","Vegan Chicken","Vegan Chicken Wrap"), ('')),
    (("Wraps","Chicken","Cranberry Chicken Wrap"), ('')),
    (("Cranberry Chicken Salad Wrap-Organic, Hormone And Antibiotic Free Chicken. Gf Wrap On Request", "", "Cranberry Chicken Wrap"), ('')),
    (("Cranberry Chicken Salad Wrap", "", "Cranberry Chicken Wrap"), ('')),
    (("Wraps", "Chickpea", "V - Smash Wrap"), (''))
    
    ]

# List the items here

In [ ]:
# Is meat but was labelled plant-based
meat = ['Cranberry Chicken Salad Wrap-Organic, Hormone And Antibiotic Free Chicken. Gf Wrap On Request',
        'Collagen Pb & J', 
        'Whipped Cream Collagen Morning Mocha',
        'Collagen Acai',
        'Collagen Pb & J Bowl',
        'Collagen Acai Smoothie Bowl',
        'Collagen Blueberry Thrill',
        'Chicken Buddha Bowl',
        'Chicken Salads',
        'Chicken Goddess Salad',
        'Chicken Sw Burrito Wrap',
        'Sushi Bowl',
        'Turkey Reuben-Vegan Or Regular',
        'Chicken Spinach Salad',
        'Banh Mi-Special For May !',
        'Chicken Apple Walnut Salad',
        'Chicken Green Goddess Salad Bowl',
        'Turkey Reuben-Vegan(Tofu) Or Rachel(Turkey)',
        'Lemon Bars',
        "Chicken Farmer'S Market Salad",
        "Shepard'S Pie & Side Salad"
        ]
# Is vegetarian but was labelled plant-based or meat
# Luke'S Cheese Puffs, 4Oz , code error - all dishes code error if have apostrophe
vegetarian = ['Veg Breakfast Sandwich',
              'Egg Avocado Toast',
              'Egg Buddha Bowl',
              'Cheese Beyond Burger',
              'Great Pumpkin Bowl (Dairy)',
              'Regular Kale! Cesar Salad',
              'Veggie Melt & Tomato Soup',
              'Veggie Melt & Tomato Basil Soup',
              'Cilantro Lime Coleslaw',
              'Egg Goddess Salad',
              'Winter Protein Salad',
              'Kid‚Äôs Organic Chocolate Milk',
              'Carrot Cake Cupcake',
              'Peanut Butter Scotchy Bar',
              'Carrot Cake Cupcake - Gf',
              'Greek Yogurt, Fruit + Granola',
              'Chips-Good Health 1Oz',
              'Peanut Butter Meltaway',
              'Frittata',
              'Peppermint Brownie',
              'Feta Toast',
              'Pretzel Bark',
              'Hard Boiled Eggs, 2, Packaged',
              'Ginger Chocolate Bar',
              'Egg Bakes',
              'Autumn Salad Bowl',
              'Solid Chocolate Bar',
              'Cake Pop (1)-Gf',
              'Creamy Tomato Soup',
              'Kettle Chips, Good Health',
              'White Cheddar Puffs',
              'Whipped Feta Toast',
              'Cauliflower Flatbread Pizza',
              'Dated Cinnamon Rolls',
              'Chocolate Peanut Butter Meltaway',
              'Potato Chips, Dirty Potato Company',
              'Quiche (Artichoke + Red Pepper) W/ Side Salad',
              'Chocolate Cake With Ganache And Raspberries, Gf',
              'Carrot Cake, Slice',
              'Caprese Focaccia Sandwich',
              'Carmel Cheesecake With Slated Caramel Sauce',
              'Majestic Pretzels',
              'Lemon Drop Cookie, Gf',
              'Chocolate Pretzel Bark',
              'Strawberry, Feta + Toasted Almond Salad',
              'Chocolate Bars: Solid, Pretzel &  Ginger',
              'Pretzel Thins',
              'Taffy',
              'Cheddar Broccoli Quiche & Side Salad  (Gf)',
              'Vanilla Layer With Cranberry Filling And Buttercream Frosting',
              'Frittata And Side Salad',
              'Cake Bites',
              'Oui Yogurt',
              'Chocolate Covered Strawberries',
              'Foiled Chocolate Carrots',
              'Sw Quesadilla',
              'Beet Feta Salad',
              'Maple Candy',
              'Frittata & Side Salad',
              'Creamy Tomato ',
              'Carrot Cake With Buttercream Frosting- V&Gt Whole Cake',
              'Carrot Cake Cupcake - Gluten Free'

]

# Is vegan but was labelled as meat?
vegan = ['Avocado Toast',
        'Vegan Breakfast Wrap',
         'Raspbalance',
         'Vegan Breakfast Bowl',
         'V - Cookie Bar',
         'V - Fudge Brownie',
         'Vegan Toasted Quesadilla',
         'Honey Pops',
         'V - Ginger Molasses Cookies',
         'V - Raspberry Lemon No Bake',
         'Chocolate Covered Oreos',
         'V - "Cheese"Cake',
         'Vegan Grilled Cheese & Tomato Soup',
         'Vegan Hot Soup',
         'Coffee Beans (12Oz Bag)',
         'Honey Lip Balm',
         'Hippea, 4Oz Puffs Siracha',
         'Vegan Cheese Melt & Soup',
         'Bath Fizzy Balls, Hawthorne 1204 Apothecary',
         '"Twix" Bar  V/Gf', # because "Twix" in quotes
         'Hippea Cheddar Puff, 4Oz',
         'Soup & Grilled Vegan Cheese',
         '24 Carrot',
         'Smash Burger + Chips',
         'Mint To Be',
         'Side Of Fries',
         'Serenity Now',
         'Peppermint Mocha Bowl',
         'Hippea, 10 Oz Cheddarpuffs',
         'Vegan Cheese Cauliflower Flatbread Pizza',
         'Microgreens, Clamshell',         
         'V - Ginger Molasses Cookies (1)',
         'Beesknees, Cinnamon Maple Syrup',
         'Cbd Full Spectrum - 1000 Mg',
         'Maple Syrup',
         'Sprout Living, 5Lb Bag, Protein Powder',
         'Sprout Living Protein Powder, 5Lb Bag',
         '5Lb. Bag Epic Protein',
         'Protein Powder, 5Lb Bag-- Sprout Living',
         'Gingersnap Cheesecake V/Gf',
         'Mints, Simply Mints',
         'Real Sport, Epic Protein Powder 1.1Lb',
         'Ginger',
         'Maple Brandy Syrup',
         'Caramel "Cheese"Cake',
         'Chickpea Salad Over Greens',
         "Vegan Farmer'S Market Salad"
         ]

# If they are not vegetarian or vegan, they can default be labeled as animal-based, so the third list is unnecessary



# Supplemental labels, unnecessary for now

non_alcoholic_drinks = [
                        'Main Squeeze', 'Calm (Formally, Main Squeeze)', 'Calm (Main Squeeze)',
                        'Orange Glow', 'Glow (Orange Glow)', 'Glow (Formally Named, Orange Glow)',
                        'Tart Smart',
                        'Bitter End',
                        'Vitali-D',                        
                        'Karma Water', 'Aqua Selzer','Aria Water','Bottled Water','Flow Water, Small','Alo Water','Voss Water','Smart Water','Alkaline Water, Large', 'C2O Water, Can', 'Aspire Water', 'Celsius Water, Can', 'Bai Water', 'Flow Water', 'Flow Water, Large',
                        'Custom Juice', # confirm if classified as 'food'
                        'Rehab', # not listed on website, not sure what it is from item mod, seems like drink
                        'Pb Scotchy, V/Gf', # not listed on site, Pb might mean peanut butter, scotchy?
                        'S Pellegrino', 'San Pellegrino, Can','Pellegrino Water, Green Bottle','Pellegrino, Can', 'Pelligrino, Glass, Large',                       
                        'Ginger Shot',                        
                        'Super Green',
                        'Immunity Booster', 'Immunity Booster - New!', 'Protect (Immunity Boost)',
                        'Aphrodite',
                        'Wheatgrass',
                        'High Voltage',
                        'Renew (Good Juju)', 'Renew (Formally Named Good Juju)',
                        'Triple C Zinger',
                        'Nourish (Salaminjaro)',
                        'Juice Box',
                        'Cold Brew Coffee', 'Ginger Tea (Shredded Ginger Root + Honey)','Chai - Housemade (No Sugar Or Dairy)','Iced Cold Brew','Spark Cold Nitro Coffee', 'Loose Leaf, Cup Of Tea','Hot Tea Bag','Dark Heart Tea','Zevia Iced Tea','Iced Chai','Wants + Needs Tea','Matcha Green Tea','Espresso','Chai: Add Sugar!','Peace Tea','Coffee, Hot Or Iced','Iced Tea','Iced Chai - Housemade (No Sugar Or Dairy)','Staff Coffee','Coffee','Chai-Free Of Sugar & Dairy','Frozen Matcha Latte', 'Iced Coffee', 'Coffee, Hot', 'Cup Of Life‚Ñ¢ Hot Tea', 'Happy Mug‚Ñ¢ Coffee', 'Iced Tea, Herbal Or Black', 'Chai', 'Chai- Dairy And Sugar Free', 'Happy Mug‚Ñ¢ Loose Leaf Tea',
                        'Celery',
                        'Glow~ Orange Glow',                        
                        'C2O Coconut Water', 'Organic Cocnunt Water, Can 11Oz','Vitacoco, Lg. Coconut Water (33.8Oz)','Coconut Water', 'Vita Coconut Water','Organic Coconut Water, Can 11Oz','Pressed Coconut Water, 1 Liter','Vita Coconut Water, 11.1 Oz','Coconut Water, Fruit Flavored','Coconut Water, 11.1 Oz', 'Coconut Water, 33.8 Oz.',
                        'Healing Lemonade',
                        'Buchi Kombucha', 'Kombucha,"Magic Hour" Orange Blossom','Aquavita Kombucha','"Hidden Worlds" Ginger Lemongrass Kombucha','"Magic Hour" Orange Blossom Kombucha','"Midnight Garden" Blueberry Lavender Kombucha', 'Kombucha, "Midnight Garden" Blueberry Lavender',
                        'Tea Bag, Prepackaged Variety',
                        'Datorade',
                        'Revive (Formally, Chlorohyllmeup)', 'Revive (Chlorohyllmeup)',
                        'Boost (Metabolic Mojito)', 'Boost (Formally, Metabolic Mojito)',
                        'Zevia Soda', 'Bubbly, Can', 'Cawston Press Soda', 'Cawston Press Rhubarb Soda',
                        'Four Sigmatic‚Ñ¢ Mushroom Cacao With Reishi',
                        'Restore (Hangover Cure)', 'Restore (Formally Named Hangover Cure)',
                        'Happy Mug Tea', 'Tea Pot And Tea',
                        '1/2 Gallon Juice, Cold Pressed',
                        'Arnold Palmer 1/2 Lemonade 1/2 Tea',
                        'Honey Tonic',
                        'Zevia Ginger Root Beer',
                        'Recover (Formally, The Roots)',
                        'Aloe Shot',
                        'Celery Juice',
                        'One Gallon Cold Pressed Juice',
                        'Grab And Go Juice',
                        '24 Carrot',
                        'Mint To Be',
                        "Dragon'S Breath"
                        ]

alcoholic_drinks = ['Bundaberg Ginger Beer',]

merch = ['Maple Syrup, Maple Brandy','Chai Meal Replacement Packet','Full Spectrum, Citrus - 1000 Mg','Extra Virgin Grapeseed Oil, Kosher','Tea Tins, Harney+Son','Vit. D3, High Potency, Capsules','Biotin Capsules, 1000Mcg','Full Spectrum - 1000 Mg','Biotin Capsules 10,000 Mcg','Elderberry Syrup 8Oz','Cbd Full Spectrum - 1000 Mg','Spicy Honey, Beesknees','Triple Magnesium Complex','Collagen Packet', 'Gum, Simply Gum', 'Coffee Beans (12Oz Bag)', 'Sprout Living Protein Packet',
         'Honey By John','Happy Mug Coffee 12Oz Bag','Spirulina Powder, Yourlixir','Elderberry Syrup, 8Oz', 'Cbd Oil', 'Cbd Full Spectrum - 500 Mg','Elderberry Drops 2Oz - Sugar-Free', 'Vitamin D, Liquid', '60 Billion Probiotic With Prebiotic',
         'Hot Cocoa Packet, Elements 4Oz', 'Simply Mints - Awaken (Caffeine)',
         'Cbd Isolate - 500 Mg', 
         'Ashwagandha Ksm-66', 'Isolate - 1000 Mg', 'Full Spectrum - 500 Mg',
         'Honey-Quart & Bear (Mv Power)' 
         ]

rare = []

unknown = ['','Enchant Mint','Honey Comb In Honey','Cure-All','Honey-Raintree Farms','Happy Gut', 'Rick Ross:  A Fall Favorite!!  (1 Sept - 15 Nov)', ]

items_to_remove = ['Towels','T Shirt - Black (Short Sleeve)','Sweatshirt','Tshirt-Light Green & Peach', 'T-Shirt- Long Sleeve, Ash White',
                   'Viridi Surface Wipes','Daily Lotion - 500 Mg',
                   'Yoga Mat Or Bug Spray ', 'Pet Wipes',
                   'Water, Regular Bottled',
                   'Flu Shot', 'Knitted Head Wraps'
                   'Drink Jar Lid, Regular',
                   'Grapeseed Oil',#?
                   'Hoodie, Juice Jar','Honey Stirrers (6 Pack)','Wooden Bamboo Spoon','Spoon, Wooden','Bath Bombs, Cupcake','Face Mask, Elestic Band', 'Lotion Bar', 'Shampoo Bar', 'Candle, Soy/Natural Andromeda', 
                   'Honey-Large Mason Quart Jar (Mv Power)','Bath Fizzy Balls, Hawthorne 1204 Apothecary', 'Honey Lip Balm',
                   'Scent Candles', 'Shampoo/Cond. Bar', 'Bath Soak','Candle, Coconut Bowl',
                   'Fruit Infusion Lid','Metal Straw','Straws And Straw Cleaning','Etched Bottle','Teapot Set','Bamboo Straw', 'Honey Bear Jar', 'Reuseable Straw', 'Drink Jar Set-Wide Mouth (Lid, Jar, Straw)','Drink Lid, Regular','Jj Nalgene  Bottle','Carafe And Tea Gift Set','Bamboo Cup And Straw Set','Drink Jar Lid, Wide Mouth','Drink Jar Set-Regular (Lid, Jar, Straw)','Four Sigmatic Golden Latte With Turkey Tail','Iron, Dbl Strength, Vegan, Now Foods','Nova Syrup - Pint','Nova Maple Syrup','Elderberry Syrup','Harmony Green Tea Packet','Sprout Living Protein Powder_Small', 'Happy Mug Coffee Beans (12Oz Bag)',
                   'Gift Certificate','Gift Card', 'Cookbook, Simple Swaps','Vegan Cookbook', 'Egift Card', '10$ Gift Certificate',
                   'Bunny Butts-Gf', 'Candle,  Andromeda'
                   'Dog Treats',
                   'Handwoven Produce Bags','Cooler Bag', 'Cooler Bags', 'Tote Bag',
                   'Magic Mud',
                   'Vapor Rub', 'Hand Cleaner, 2Oz., Natural, Alcohol-Free',
                   'Merchandise', 'Clock', 
                   'Candles, Glass Jar, Big Moods'
                   ] # nonfood

In [ ]:
df_relabeled = fully_relabel_and_consolidate(df_uncleaned,
                                             remove=items_to_remove,
                                             modification_name_changes=list(zip(*modification_name_changes))[0],
                                             vegan_list=vegan,
                                             vegetarian_list=vegetarian,
                                             meat_list=meat,
                                             drinks_list=non_alcoholic_drinks,
                                             alcohol_list=alcoholic_drinks,
                                             merch=merch,
                                             rare=rare,
                                             unknown=unknown
                                             )

df_relabeled.to_parquet(f"data/4_palate_data_parquet_relabeled/relabeled/{loc_id}_sales_and_menu.parquet")

df_consolidated = df_relabeled.pipe(rename_items, name_changes = {})

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)

df_consolidated.to_parquet(f"data/4_palate_data_parquet_relabeled/consolidated/{loc_id}_sales_and_menu.parquet")

In [ ]:
plot_dish_time_series(df_consolidated
                      .dropna(subset='item_modifications')
                      .query('item_name.str.contains("Desserts") and item_modifications.str.contains("Vegan")')
                      .assign(item_name = lambda df: df['item_modifications']),
                      loc_id,
                      before_after_details_true,
                      top_n=30)

In [ ]:
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cheesecake")')[['item_name','item_modifications']])

In [ ]:
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cupcake")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Gf Cake Slice")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cheesecake")').index[3])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Bar, Vegan")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_name.str.contains("Vegan Breakfast Sandwich")').index[0])

In [ ]:
plot_dish_time_series(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan")'), loc_id, before_after_details_true, top_n=30)

In [ ]:
vegan_str = before_after_details_true.loc[loc_id,'promo_name'][0]
bacon_str = before_after_details_true.loc[loc_id,'promo_name'][1]
promo_item_containing = df_uncleaned.loc[lambda df: df['item_name'].str.contains("Vegan Breakfast Sandwich", na=False)]
promo_item_containing_2 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cheese", na=False)]
promo_item_containing_3 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Unchicken|Un'chicken", na=False)]
promo_item_containing_4 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cupcake", na=False)]
promo_item_containing_5 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cupcake", na=False)]
promo_item_containing_6 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Sausage", na=False)]
promo_item_containing_7 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cheese|Vegan Cheddar", na=False)]
awkward_aardvark = df_uncleaned.loc[lambda df: df['item_name'].str.contains("Awkward Aardvark", na=False)]
print(promo_item_containing['item_name'].unique().tolist())

plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_2.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_3.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_4.resample('W')['item_quantity'].sum())
#plt.plot(promo_item_containing_5.resample('W')['item_quantity'].sum())
#plt.plot(promo_item_containing_6.resample('W')['item_quantity'].sum())
#plt.plot(promo_item_containing_7.resample('W')['item_quantity'].sum())

plt.plot(df_uncleaned.resample('W')['item_quantity'].sum())
plt.axvline(x=before_after_details_true.loc[loc_id, 'cross_over_date'], color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
#Legend
plt.legend([
    'Vegan Breakfast Sandwich',
    'Vegan Cheese',
    'Unchicken',
    'Vegan Cupcake',
    'All',
    #'Vegan Sausage',
    #'Vegan Cheese'
])
plt.xticks(rotation=50)
plt.show()

In [ ]:
clean_df.query('item_name.str.contains("Smash")')[['item_modifications']].value_counts()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plot_time_series(loc_id, 
                 df_uncleaned.dropna(subset='item_modifications').query('item_name.str.contains("V - Smash Wrap")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
clean_df.query('item_name == "Wraps"')[['item_name','item_modifications','is_plant_based']].value_counts()

In [ ]:
plot_dish_time_series(clean_df, loc_id, before_after_details_true)